# J-Space Experiment — Phases 1–4 (Local Linux)

End-to-end launcher for a local Linux machine with CUDA. Each phase runs through the same CLI used from the terminal. Results are saved under one run root inside this repository.

Launch from the repository root so `uv` can create and use the project virtualenv:

```bash
uv run jupyter notebook notebooks/JSpace_End_to_End_Local.ipynb
```

The first code cell runs `uv sync` and verifies that this notebook kernel is using `.venv/`. Set `OPENROUTER_API_KEY` before Phase 2/4 analysis and authenticate with Hugging Face (`hf auth login` or `HF_TOKEN`).

## 1. Sync the uv environment and verify pinned checkouts

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

AGENTDOJO_REVISION = '089ed468cf3ed0322acc66b0211f26d9d90dbf60'
INJECAGENT_REVISION = 'f19c9f2c79a41046eb13c03c51a24c567a8ffa07'
UV_SYNC_EXTRAS = ('phase4', 'notebook')


def require_uv() -> None:
    if shutil.which('uv') is None:
        raise RuntimeError(
            'uv is not installed. Install it from https://docs.astral.sh/uv/getting-started/installation/'
        )


def sync_uv_environment(repo_root: Path) -> Path:
    require_uv()
    subprocess.run(
        ['uv', 'sync', *(arg for extra in UV_SYNC_EXTRAS for arg in ('--extra', extra))],
        cwd=repo_root,
        check=True,
    )
    venv_python = (repo_root / '.venv' / 'bin' / 'python').resolve()
    if not venv_python.is_file():
        raise RuntimeError(f'uv sync did not create a virtualenv at {venv_python}')
    return venv_python


def verify_notebook_kernel(venv_python: Path) -> None:
    current_python = Path(sys.executable).resolve()
    if current_python != venv_python:
        raise RuntimeError(
            'This notebook is not running in the project virtualenv.\n'
            f'Expected: {venv_python}\n'
            f'Current:  {current_python}\n'
            'Restart from the repository root with:\n'
            '  uv run jupyter notebook notebooks/JSpace_End_to_End_Local.ipynb'
        )


def verify_project_environment(repo_root: Path) -> None:
    subprocess.run(
        [
            'uv',
            'run',
            'python',
            '-c',
            (
                'import jspace_research, pandas, torch; '
                'from jspace_research.phase1.cli import main; '
                'print("environment ok")'
            ),
        ],
        cwd=repo_root,
        check=True,
    )
    subprocess.run(
        ['uv', 'run', 'jspace-phase1', '--help'],
        cwd=repo_root,
        check=True,
        stdout=subprocess.DEVNULL,
    )


cwd = Path.cwd().resolve()
REPO_ROOT = cwd if (cwd / 'pyproject.toml').is_file() else cwd.parent
if not (REPO_ROOT / 'pyproject.toml').is_file():
    raise RuntimeError('Run this notebook from the repository root or notebooks/ directory.')

VENV_PYTHON = sync_uv_environment(REPO_ROOT)
verify_notebook_kernel(VENV_PYTHON)
verify_project_environment(REPO_ROOT)

BENCHMARKS_ROOT = Path(
    os.environ.get('JSPACE_BENCHMARKS_ROOT', REPO_ROOT.parent / 'jspace-benchmarks')
).expanduser().resolve()
BIPIA_CHECKOUT = REPO_ROOT / 'BIPIA'
AGENTDOJO_CHECKOUT = BENCHMARKS_ROOT / 'agentdojo'
INJECAGENT_CHECKOUT = BENCHMARKS_ROOT / 'InjecAgent'

subprocess.run(['git', 'submodule', 'update', '--init', 'BIPIA'], cwd=REPO_ROOT, check=True)

BENCHMARKS_ROOT.mkdir(parents=True, exist_ok=True)
if not AGENTDOJO_CHECKOUT.exists():
    subprocess.run(
        ['git', 'clone', 'https://github.com/ethz-spylab/agentdojo.git', str(AGENTDOJO_CHECKOUT)],
        check=True,
    )
subprocess.run(['git', '-C', str(AGENTDOJO_CHECKOUT), 'checkout', AGENTDOJO_REVISION], check=True)
if not INJECAGENT_CHECKOUT.exists():
    subprocess.run(
        ['git', 'clone', 'https://github.com/uiuc-kang-lab/InjecAgent.git', str(INJECAGENT_CHECKOUT)],
        check=True,
    )
subprocess.run(['git', '-C', str(INJECAGENT_CHECKOUT), 'checkout', INJECAGENT_REVISION], check=True)

print('Repository root:', REPO_ROOT)
print('Virtualenv python:', VENV_PYTHON)
print('Benchmarks root:', BENCHMARKS_ROOT)
print('Research revision:', subprocess.check_output(['git', '-C', str(REPO_ROOT), 'rev-parse', 'HEAD'], text=True).strip())
print('BIPIA revision:', subprocess.check_output(['git', '-C', str(BIPIA_CHECKOUT), 'rev-parse', 'HEAD'], text=True).strip())
print('AgentDojo revision:', subprocess.check_output(['git', '-C', str(AGENTDOJO_CHECKOUT), 'rev-parse', 'HEAD'], text=True).strip())
print('InjecAgent revision:', subprocess.check_output(['git', '-C', str(INJECAGENT_CHECKOUT), 'rev-parse', 'HEAD'], text=True).strip())

## 2. Authenticate

In [ ]:
import os

from huggingface_hub import login

hf_token = os.environ.get('HF_TOKEN') or os.environ.get('HUGGING_FACE_HUB_TOKEN')
if hf_token:
    login(token=hf_token, add_to_git_credential=False)
    print('Hugging Face token loaded from the environment.')
else:
    login(add_to_git_credential=False)
    print('Hugging Face login complete.')

if not os.environ.get('OPENROUTER_API_KEY'):
    raise RuntimeError(
        'Set OPENROUTER_API_KEY in the environment before Phase 2/4 analysis. '
        'Example: export OPENROUTER_API_KEY=...'
    )
print('OpenRouter judge credential is set.')

## 3. Configure one persistent run directory

In [ ]:
RUN_MODE = 'smoke'  # use 'full' only after smoke succeeds
RUN_NAME = f'jspace-{RUN_MODE}'

RUN_ROOT = REPO_ROOT / 'artifacts' / RUN_NAME
PHASE1_DIR = RUN_ROOT / 'phase1'
PHASE2_DIR = RUN_ROOT / 'phase2'
PHASE3_DIR = RUN_ROOT / 'phase3'
PHASE4_DIR = RUN_ROOT / 'phase4'
BIPIA_ROOT = BIPIA_CHECKOUT / 'benchmark'
CONFIG_PATH = REPO_ROOT / 'configs' / f'phase1_{RUN_MODE}.yaml'
WEBQA_TRAIN_PATH = None  # required for full mode
SUMMARIZATION_TRAIN_PATH = None  # required for full mode


def run_command(command, label):
    if command and command[0].startswith('jspace-'):
        command = ['uv', 'run', *command]
    print('Running:', ' '.join(str(part) for part in command))
    process = subprocess.Popen(
        command,
        cwd=REPO_ROOT,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='')
    return_code = process.wait()
    if return_code != 0:
        raise RuntimeError(f'{label} failed with exit status {return_code}; see the traceback above.')


RUN_ROOT.mkdir(parents=True, exist_ok=True)
print('Config:', CONFIG_PATH)
print('Run root:', RUN_ROOT)

## 4. Verify the GPU runtime

In [ ]:
import torch

assert torch.cuda.is_available(), 'Phase 1 capture/analyze and Phase 2/4 generate require a CUDA GPU.'
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0))

## 5. Run or resume Phase 1

This freezes the manifest, captures activations, reconstructs J-space, and selects the layer. Rerunning the cell reuses compatible caches.

In [ ]:
phase1_command = [
    'jspace-phase1',
    '--config', str(CONFIG_PATH),
    '--output-dir', str(PHASE1_DIR),
    '--stage', 'all',
]
if WEBQA_TRAIN_PATH is not None:
    phase1_command.extend(['--webqa-train', str(WEBQA_TRAIN_PATH)])
if SUMMARIZATION_TRAIN_PATH is not None:
    phase1_command.extend(['--summarization-train', str(SUMMARIZATION_TRAIN_PATH)])
run_command(phase1_command, 'Phase 1')

## 6. Inspect Phase 1 before continuing

In [ ]:
import json

import pandas as pd
from IPython.display import Image, display

selection = json.loads((PHASE1_DIR / 'selected_layer.json').read_text())
print(json.dumps(selection, indent=2))
display(pd.read_csv(PHASE1_DIR / 'layer_metrics.csv'))
display(Image(filename=str(PHASE1_DIR / 'layer_auprc.png')))
display(Image(filename=str(PHASE1_DIR / 'selected_layer_score_distribution.png')))

## 7. Run or resume Phase 2 generation

This GPU stage reads the frozen Phase 1 directory directly and runs three conditions: intact (`alpha=0.0`), partial removal (`alpha=0.5`), and full removal (`alpha=1.0`).

In [ ]:
phase2_base = [
    'jspace-phase2',
    '--config', str(CONFIG_PATH),
    '--phase1', str(PHASE1_DIR / 'selected_layer.json'),
    '--output-dir', str(PHASE2_DIR),
]
run_command([*phase2_base, '--stage', 'generate'], 'Phase 2 generation')

## 8. Run or resume Phase 2 analysis

This stage uses cached generations, ROUGE scoring, and the pinned OpenRouter judge.

In [ ]:
run_command([*phase2_base, '--stage', 'analyze'], 'Phase 2 analysis')

## 9. Inspect Phase 2 results

In [ ]:
display(pd.read_csv(PHASE2_DIR / 'phase2_summary.csv'))
display(pd.read_csv(PHASE2_DIR / 'phase2_examples.csv'))
display(Image(filename=str(PHASE2_DIR / 'phase2_asr_vs_alpha.png')))
display(Image(filename=str(PHASE2_DIR / 'phase2_clean_utility_vs_alpha.png')))

## 10. Construct and inspect the Phase 3 detectors

This CPU-only stage reads the frozen Phase 1 handoff directly. It does not load Gemma or the lens and does not depend on Phase 2.

In [ ]:
phase3_command = [
    'jspace-phase3',
    '--config', str(CONFIG_PATH),
    '--phase1', str(PHASE1_DIR / 'selected_layer.json'),
    '--output-dir', str(PHASE3_DIR),
]
run_command(phase3_command, 'Phase 3')
display(pd.read_csv(PHASE3_DIR / 'phase3_metrics.csv'))
display(Image(filename=str(PHASE3_DIR / 'phase3_detector_comparison.png')))

## 11. Run or resume and inspect Phase 4

This runs the three frozen transfer benchmarks. Generation requires CUDA; analysis uses CPU and OpenRouter only for BIPIA semantic outcomes.

In [ ]:
phase4_base = [
    'jspace-phase4',
    '--config', str(CONFIG_PATH),
    '--phase1', str(PHASE1_DIR / 'selected_layer.json'),
    '--phase3', str(PHASE3_DIR),
    '--bipia-root', str(BIPIA_ROOT),
    '--agentdojo-root', str(AGENTDOJO_CHECKOUT),
    '--injecagent-root', str(INJECAGENT_CHECKOUT),
    '--output-dir', str(PHASE4_DIR),
]
run_command([*phase4_base, '--stage', 'generate'], 'Phase 4 generation')
run_command([*phase4_base, '--stage', 'analyze'], 'Phase 4 analysis')
display(pd.read_csv(PHASE4_DIR / 'phase4_metrics.csv'))
display(Image(filename=str(PHASE4_DIR / 'phase4_detector_transfer.png')))

## 12. Confirm persistence

All caches and results are written under the local run root. Preserve the `phase1/`, `phase2/`, `phase3/`, and `phase4/` directories together when copying or archiving a run.

In [ ]:
print('Complete run root:', RUN_ROOT)
print('Phase 1 selected layer:', selection['selected_layer'])
print('Phase 2 results:', PHASE2_DIR / 'phase2_results.parquet')
print('Phase 3 metrics:', PHASE3_DIR / 'phase3_metrics.csv')
print('Phase 4 metrics:', PHASE4_DIR / 'phase4_metrics.csv')
assert (PHASE1_DIR / 'selected_layer.json').is_file()
assert (PHASE2_DIR / 'phase2_results.parquet').is_file()
assert (PHASE3_DIR / 'mean_detector.pt').is_file()
assert (PHASE3_DIR / 'logistic_detector.pt').is_file()
assert (PHASE4_DIR / 'phase4_predictions.parquet').is_file()

## Interpretation boundary

An effect in Phase 2 shows that the selected layer's reconstructed J-space component is functionally involved in model behavior. Phase 3 measures development-set detectability and freezes thresholds. Phase 4 evaluates those frozen detectors on held-out and transfer benchmarks without tuning. None of these phases establishes injection-specific causality.